In [0]:
df_fact = spark.read.table("abn_demo.silver_abn.fact_table")


In [0]:
from pyspark.sql.functions import count 

episodes_per_season = (
    df_fact
    .select("show_id", "show_name", "season", "episode_name") 
    .groupBy("show_id", "show_name", "season")
    .agg(count("episode_name").alias("episode_count"))
)

episodes_per_season.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("abn_demo.gold_abn.episodes_per_season")

In [0]:

avg_runtime = df_fact.select("show_id", "show_name", "season", "episode_name", "runtime")\
    .groupBy("show_id", "show_name")\
    .avg("runtime") \
    .withColumnRenamed("avg(runtime)", "avg_runtime")

avg_runtime.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("gold_abn.avg_runtime")


In [0]:

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window = Window.partitionBy("show_id").orderBy(desc("count"))
top_cast = df_fact.groupBy("show_id", "cast_name") \
    .count() \
    .withColumn("rank", row_number().over(window)) \
    .filter("rank <= 3")\
    .drop("rank","count")

top_cast.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("gold_abn.top_cast")


In [0]:
common_genres = df_fact.groupBy("genre") \
    .count() \
    .orderBy("count", ascending=False)

common_genres.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("gold_abn.common_genres")